# Targeted XGBoost Tuning

## Objectives
- Focus only on the strongest model family from notebook 4: XGBoost trained on the original scaled data.
- Run a tighter anti-overfitting hyperparameter search around the best-performing region.
- Tune the decision threshold on validation predictions and evaluate the result on the held-out test split.
- Save focused search outputs to `../data/` for easy comparison with notebook 4.


In [ ]:
# Mount Google Drive (only needed in Google Colab)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Set the working directory (only needed in Google Colab)
import os
os.chdir('/content/drive/MyDrive/bitcoin-ransomware-classifier/notebooks')
print('Working directory:', os.getcwd())


## Load Prepared Splits

This section loads the original scaled training split from notebook 2, along with the untouched validation and test splits. The goal is to stay on the training recipe that generalized best in the broader notebook 4 search.


In [ ]:
import pandas as pd

X_train = pd.read_csv('../data/X_train_original.csv')
X_val = pd.read_csv('../data/X_val.csv')
X_test = pd.read_csv('../data/X_test.csv')

y_train = pd.read_csv('../data/y_train_original.csv').squeeze('columns')
y_val = pd.read_csv('../data/y_val.csv').squeeze('columns')
y_test = pd.read_csv('../data/y_test.csv').squeeze('columns')

imbalance_ratio = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))

print('X_train shape:', X_train.shape)
print('X_val shape:', X_val.shape)
print('X_test shape:', X_test.shape)
print('Original train neg:pos ratio:', f'{imbalance_ratio:.2f}:1')


## XGBoost Setup

This section configures the focused XGBoost search. The search space stays close to the current winner by favoring shallow trees, strong regularization, moderate subsampling, and imbalance-aware weighting.


In [ ]:
import numpy as np
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.metrics import balanced_accuracy_score, f1_score, precision_score, recall_score

GPU_ENABLED = False

try:
    from xgboost import XGBClassifier
    GPU_ENABLED = True
    print('XGBoost available. CUDA will be used when supported by the Colab runtime.')
except Exception as exc:
    GPU_ENABLED = False
    raise RuntimeError(f'XGBoost is required for this notebook: {exc}')


def score_predictions(y_true, y_pred):
    return {
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
    }


def normalize_predictions(y_pred):
    if hasattr(y_pred, 'to_pandas'):
        y_pred = y_pred.to_pandas()
    elif type(y_pred).__module__.startswith('cupy'):
        y_pred = y_pred.get()
    return np.asarray(y_pred)


def normalize_scores(values):
    if hasattr(values, 'to_pandas'):
        values = values.to_pandas()
    elif type(values).__module__.startswith('cupy'):
        values = values.get()
    values = np.asarray(values)
    if values.ndim == 2 and values.shape[1] > 1:
        values = values[:, 1]
    return values.reshape(-1)


xgb_params = {
    'random_state': 12,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
}
if GPU_ENABLED:
    xgb_params['device'] = 'cuda'

scale_pos_weight_grid = sorted({round(max(1.0, imbalance_ratio * factor), 2) for factor in [0.85, 1.0, 1.15]})

search_space = {
    'n_estimators': [400, 600, 800, 1000],
    'learning_rate': [0.02, 0.03, 0.05],
    'max_depth': [3, 4],
    'min_child_weight': [10, 15, 20],
    'subsample': [0.65, 0.7, 0.75],
    'colsample_bytree': [0.5, 0.6, 0.7],
    'gamma': [2.0, 3.0, 5.0],
    'reg_alpha': [0.5, 1.0, 3.0, 5.0],
    'reg_lambda': [10.0, 15.0, 20.0, 30.0],
    'scale_pos_weight': scale_pos_weight_grid,
}

print('scale_pos_weight grid:', scale_pos_weight_grid)


## Focused Randomized Search

This section uses the existing train/validation split from the project workflow. It keeps the search budget on one model family instead of spending runtime on weaker models.


In [ ]:
X_search = pd.concat([X_train, X_val], ignore_index=True)
y_search = pd.concat([y_train, y_val], ignore_index=True)
validation_fold = [-1] * len(X_train) + [0] * len(X_val)
splitter = PredefinedSplit(test_fold=validation_fold)

search = RandomizedSearchCV(
    estimator=XGBClassifier(**xgb_params),
    param_distributions=search_space,
    n_iter=20,
    scoring='f1',
    cv=splitter,
    random_state=12,
    verbose=1,
    refit=True,
    return_train_score=True,
)

search.fit(X_search, y_search)

search_results = pd.DataFrame(search.cv_results_)
search_results['model'] = 'xgb'
search_results['train_source'] = 'Original'
search_results['feature_set'] = 'Full'
search_results['overfit_gap'] = search_results['mean_train_score'] - search_results['mean_test_score']
search_results['params'] = search_results['params'].apply(lambda params: dict(params))

search_results[['rank_test_score', 'mean_test_score', 'overfit_gap', 'params']].sort_values('rank_test_score').head(10)


## Best Validation Result

This section captures the best focused XGBoost configuration and scores it on the validation split with the default threshold before any threshold tuning.


In [ ]:
best_model = search.best_estimator_
y_val_pred = normalize_predictions(best_model.predict(X_val))

best_validation_model = pd.DataFrame([
    {
        'model': 'xgb',
        'train_source': 'Original',
        'feature_set': 'Full',
        'params': dict(search.best_params_),
        'overfit_gap': float(search_results.loc[search_results['rank_test_score'].idxmin(), 'overfit_gap']),
        **score_predictions(y_val, y_val_pred),
    }
])

best_validation_model


## Test Set Evaluation

This section measures how the best focused XGBoost model performs on the held-out test split before threshold tuning.


In [ ]:
y_test_pred = normalize_predictions(best_model.predict(X_test))

test_results = pd.DataFrame([
    {
        'model': 'xgb',
        'train_source': 'Original',
        'feature_set': 'Full',
        'params': dict(search.best_params_),
        'overfit_gap': float(search_results.loc[search_results['rank_test_score'].idxmin(), 'overfit_gap']),
        **score_predictions(y_test, y_test_pred),
    }
])

test_results


## Threshold Tuning

This section sweeps thresholds on validation probabilities, refines around the strongest range, and then applies the best threshold to the test set without retraining the model.


In [ ]:
def evaluate_thresholds(y_true, scores, thresholds):
    rows = []
    for threshold in thresholds:
        y_pred = (scores >= threshold).astype(int)
        rows.append({
            'threshold': float(threshold),
            **score_predictions(y_true, y_pred),
        })
    return pd.DataFrame(rows)


def find_best_threshold(y_true, scores):
    coarse_thresholds = np.linspace(0.60, 0.98, 77)
    coarse_df = evaluate_thresholds(y_true, scores, coarse_thresholds)
    coarse_df = coarse_df.sort_values(['f1', 'balanced_accuracy'], ascending=False).reset_index(drop=True)

    coarse_best = float(coarse_df.loc[0, 'threshold'])
    fine_min = max(0.50, coarse_best - 0.04)
    fine_max = min(0.995, coarse_best + 0.04)
    fine_thresholds = np.linspace(fine_min, fine_max, 81)

    threshold_df = pd.concat([coarse_df, evaluate_thresholds(y_true, scores, fine_thresholds)], ignore_index=True)
    threshold_df = threshold_df.drop_duplicates(subset=['threshold']).sort_values(['f1', 'balanced_accuracy'], ascending=False).reset_index(drop=True)
    return threshold_df, threshold_df.iloc[0].to_dict()


val_scores = normalize_scores(best_model.predict_proba(X_val))
threshold_search_results, best_threshold_row = find_best_threshold(y_val, val_scores)
threshold_search_results['model'] = 'xgb'
threshold_search_results['train_source'] = 'Original'
threshold_search_results['feature_set'] = 'Full'
threshold_search_results['params'] = str(dict(search.best_params_))
threshold_search_results['overfit_gap'] = float(search_results.loc[search_results['rank_test_score'].idxmin(), 'overfit_gap'])

best_threshold = float(best_threshold_row['threshold'])
test_scores = normalize_scores(best_model.predict_proba(X_test))
threshold_test_pred = (test_scores >= best_threshold).astype(int)

threshold_test_results = pd.DataFrame([
    {
        'model': 'xgb',
        'train_source': 'Original',
        'feature_set': 'Full',
        'params': dict(search.best_params_),
        'overfit_gap': float(search_results.loc[search_results['rank_test_score'].idxmin(), 'overfit_gap']),
        'threshold': best_threshold,
        **score_predictions(y_test, threshold_test_pred),
    }
]).sort_values(['f1', 'balanced_accuracy'], ascending=False).reset_index(drop=True)

threshold_test_results


## Save Focused Outputs

This section writes the focused XGBoost search summaries back to `../data/` so they can be compared directly with the broader notebook 4 outputs.


In [ ]:
search_results.to_csv('../data/xgb_targeted_search_results.csv', index=False)
best_validation_model.to_csv('../data/xgb_targeted_best_validation_model.csv', index=False)
test_results.to_csv('../data/xgb_targeted_test_results.csv', index=False)
threshold_search_results.to_csv('../data/xgb_targeted_threshold_search_results.csv', index=False)
threshold_test_results.to_csv('../data/xgb_targeted_threshold_test_results.csv', index=False)

print('Saved focused XGBoost outputs to ../data/')
